# Laboratorul 5 — Transformers
## De la atenție la generarea de text

**Disciplina:** Inteligență Artificială Generativă  
**Student:** Carp Dan  
**Mediu recomandat:** Jupyter Notebook sau Google Colab  
**Timp estimat:** 2–3 ore

În acest laborator construiesc intuiția pentru arhitectura Transformer pornind de la atenția calculată cu NumPy și continuând cu tokenizere și modele preantrenate. Experimentele urmăresc: atenția scaled dot-product, mascarea cauzală și multi-head attention; tokenizarea textelor românești; predicția și generarea autoregresivă; reprezentări contextuale BERT și hărți de atenție.

**Notă:** celulele sunt pregătite pentru rulare, dar rezultatele depind de versiunile bibliotecilor și de modelele descărcate. Completez interpretările după ce rulez experimentele; nu presupun rezultate care nu au fost măsurate.

## Obiective și concepte-cheie

La finalul laboratorului pot:

1. explica rolul interogărilor (Q), cheilor (K) și valorilor (V) și implementa formula atenției;
2. explica de ce scorurile sunt împărțite la $\sqrt{d_k}$ și cum funcționează masca autoregresivă;
3. diferenția tokenizarea de împărțirea simplă în cuvinte și compara tokenizere pentru română;
4. inspecta distribuția următorului token și compara decodarea greedy cu sampling la temperaturi diferite;
5. compara reprezentări contextuale ale aceluiași cuvânt în contexte diferite și explora atenția unui model BERT.

### Formulele folosite

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+M\right)V$$

Aici $M$ este opțională: pentru pozițiile interzise într-o atenție cauzală, valoarea adăugată este $-\infty$. Pentru sampling cu temperatură $T>0$, probabilitățile următorului token sunt $p_i=\operatorname{softmax}(z_i/T)$, unde $z_i$ sunt scorurile (logits) modelului.

## 0. Pregătirea mediului

Modelele sunt preantrenate și se descarcă la prima utilizare. Laboratorul nu antrenează un model. CPU este suficient pentru exemplele mici, însă încărcarea și inferența pot dura; un GPU este opțional. Celula următoare instalează dependențele în notebook.

In [ ]:
%pip -q install "numpy>=1.24" "pandas>=2.0" "matplotlib>=3.7" "torch>=2.0" "transformers>=4.40,<5" "bertviz>=1.4"

In [ ]:
import math
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"NumPy: {np.__version__} | PyTorch: {torch.__version__}")
print(f"Dispozitiv de inferență: {device}")

---
# 1. Atenția calculată cu NumPy

Un token folosește un vector Query pentru a exprima ce informație caută, Keys pentru a compara relevanța celorlalte poziții și Values pentru a combina informația selectată. În implementările Transformer, proiecțiile Q, K și V sunt învățate; în această secțiune le construiesc pe date mici pentru a urmări calculul.

## 1.1 Softmax stabil numeric

Softmax transformă scoruri arbitrare în ponderi pozitive a căror sumă este 1. Scăderea maximului înainte de exponentiere nu schimbă distribuția normalizată și previne overflow-ul pentru scoruri mari.

In [ ]:
def softmax(scores, axis=-1):
    scores = np.asarray(scores, dtype=np.float64)
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=axis, keepdims=True)


scores = np.array([1.2, 2.8, 0.7])
weights = softmax(scores)
print("Scoruri:", scores)
print("Ponderi Softmax:", weights)
print("Suma ponderilor:", weights.sum())
assert np.isclose(weights.sum(), 1.0)

large_scores = np.array([1000.0, 1001.0])
large_weights = softmax(large_scores)
print("Ponderi pentru scoruri mari:", large_weights)
assert np.all(np.isfinite(large_weights)) and np.isclose(large_weights.sum(), 1.0)

**Interpretare de completat după rulare:** compar rezultatul pentru scorurile obișnuite cu cel pentru scorurile mari. De ce varianta stabilă rămâne finită? Ce proprietate a distribuției este verificată de suma ponderilor?

## 1.2 Scaled dot-product attention

Calculez $QK^T$, scalez scorurile cu $\sqrt{d_k}$, aplic Softmax și formez suma ponderată a valorilor. Opțional, masca booleană indică pozițiile vizibile (`True`).

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Returnează ieșirea atenției și ponderile, pentru matrice 2D."""
    Q = np.asarray(Q, dtype=np.float64)
    K = np.asarray(K, dtype=np.float64)
    V = np.asarray(V, dtype=np.float64)
    if Q.ndim != 2 or K.ndim != 2 or V.ndim != 2:
        raise ValueError("Q, K și V trebuie să fie matrice bidimensionale.")
    if Q.shape[1] != K.shape[1] or K.shape[0] != V.shape[0]:
        raise ValueError("Dimensiunile Q/K/V nu sunt compatibile.")

    scores = (Q @ K.T) / np.sqrt(K.shape[1])
    if mask is not None:
        mask = np.asarray(mask, dtype=bool)
        if mask.shape != scores.shape:
            raise ValueError("Masca trebuie să aibă aceeași formă ca matricea scorurilor.")
        if np.any(~mask.all(axis=1) & ~mask.any(axis=1)):
            raise ValueError("Fiecare rând trebuie să aibă cel puțin o poziție vizibilă.")
        scores = np.where(mask, scores, -np.inf)

    weights = softmax(scores, axis=-1)
    return weights @ V, weights


# O interogare și trei perechi Key/Value, într-un exemplu de dimensiune mică.
Q = np.array([[2.0, 1.0, 0.0, 1.0]])
K = np.array([
    [0.5, 1.0, 0.3, 0.4],
    [2.0, 1.0, 0.5, 0.6],
    [0.2, 0.6, 0.9, 0.4],
])
V = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])

context, attention_weights = scaled_dot_product_attention(Q, K, V)
print("Scoruri:", (Q @ K.T) / np.sqrt(K.shape[1]))
print("Ponderi:", attention_weights)
print("Vectorul contextualizat:", context)
assert np.allclose(attention_weights.sum(axis=-1), 1.0)
assert context.shape == (1, V.shape[1])

**Întrebare:** care Value contribuie cel mai mult la rezultatul contextualizat în acest exemplu? Explic răspunsul folosind ponderile obținute, nu o presupunere despre semnificația vectorilor.

## 1.3 De ce împărțim la $\sqrt{d_k}$?

Când dimensiunea vectorilor crește, produsele scalare tind să aibă magnitudini mai mari. Aplicarea Softmax direct pe scoruri foarte depărtate poate concentra ponderile într-o singură poziție. Următorul experiment compară concentrarea pentru scoruri scalate și nescalate, pe date generate cu o sămânță fixă.

In [ ]:
rng = np.random.default_rng(23)
dimension_rows = []
for d_k in (4, 16, 64, 256):
    q = rng.normal(size=(80, d_k))
    k = rng.normal(size=(80, d_k))
    raw = q @ k.T
    scaled = raw / np.sqrt(d_k)
    dimension_rows.append({
        "d_k": d_k,
        "pondere maximă medie, scalat": softmax(scaled).max(axis=-1).mean(),
        "pondere maximă medie, nescalat": softmax(raw).max(axis=-1).mean(),
    })

scaling_table = pd.DataFrame(dimension_rows)
display(scaling_table)

**Interpretare de completat:** urmăresc schimbarea ponderii maxime medii pe măsură ce crește $d_k$. Explic ce efect are scalarea asupra distribuției și de ce poate ajuta optimizarea.

## 1.4 Atenția cauzală

Un model autoregresiv nu trebuie să vadă tokenii viitori când prezice următorul token. Construiesc o mască triunghiulară inferioară: poziția $i$ poate participa doar la cheile de la 0 la $i$. Elementele viitoare sunt eliminate înainte de Softmax.

In [ ]:
rng = np.random.default_rng(42)
sequence_length, model_dim, key_dim, value_dim = 5, 8, 4, 6
X = rng.normal(size=(sequence_length, model_dim))
W_Q = rng.normal(size=(model_dim, key_dim))
W_K = rng.normal(size=(model_dim, key_dim))
W_V = rng.normal(size=(model_dim, value_dim))
Q, K, V = X @ W_Q, X @ W_K, X @ W_V

causal_mask = np.tril(np.ones((sequence_length, sequence_length), dtype=bool))
causal_output, causal_weights = scaled_dot_product_attention(Q, K, V, causal_mask)
print("Mască (True = vizibil):\n", causal_mask.astype(int))
print("Ponderi de atenție:\n", causal_weights)
print("Suma fiecărui rând:", causal_weights.sum(axis=1))
assert np.allclose(causal_weights.sum(axis=1), 1.0)
assert np.allclose(causal_weights[np.triu_indices(sequence_length, k=1)], 0.0)
assert np.allclose(causal_weights[0], [1.0, 0.0, 0.0, 0.0, 0.0])

**Întrebare:** de ce primul rând al matricei de atenție are pondere nenulă doar pentru prima poziție? Ce eroare de predicție s-ar putea produce dacă masca ar permite accesul la tokeni viitori în antrenarea unui model autoregresiv?

## 1.5 Multi-head attention

Multi-head attention rulează mai multe mecanisme de atenție în paralel, fiecare cu propriile proiecții, concatenează ieșirile și aplică o proiecție de ieșire. În exemplul de mai jos implementez două capete mici, cu mască cauzală comună.

In [ ]:
def multi_head_self_attention(X, num_heads=2, mask=None, seed=5):
    X = np.asarray(X, dtype=np.float64)
    if X.ndim != 2 or X.shape[1] % num_heads != 0:
        raise ValueError("Dimensiunea reprezentării trebuie să fie divizibilă cu num_heads.")
    model_dim = X.shape[1]
    head_dim = model_dim // num_heads
    rng = np.random.default_rng(seed)
    W_q = rng.normal(scale=1 / np.sqrt(model_dim), size=(num_heads, model_dim, head_dim))
    W_k = rng.normal(scale=1 / np.sqrt(model_dim), size=(num_heads, model_dim, head_dim))
    W_v = rng.normal(scale=1 / np.sqrt(model_dim), size=(num_heads, model_dim, head_dim))
    W_o = rng.normal(scale=1 / np.sqrt(model_dim), size=(model_dim, model_dim))

    head_outputs, head_weights = [], []
    for head in range(num_heads):
        q, k, v = X @ W_q[head], X @ W_k[head], X @ W_v[head]
        output, weights = scaled_dot_product_attention(q, k, v, mask)
        head_outputs.append(output)
        head_weights.append(weights)

    concatenated = np.concatenate(head_outputs, axis=-1)
    return concatenated @ W_o, np.stack(head_weights)


multi_head_output, per_head_weights = multi_head_self_attention(
    X, num_heads=2, mask=causal_mask
)
print("Formă ieșire:", multi_head_output.shape)
print("Formă ponderi [cap, query, key]:", per_head_weights.shape)
assert multi_head_output.shape == X.shape
assert np.allclose(per_head_weights.sum(axis=-1), 1.0)
assert np.allclose(per_head_weights[:, np.triu_indices(sequence_length, k=1)[0],
                                    np.triu_indices(sequence_length, k=1)[1]], 0.0)

**Interpretare:** compar tiparele celor două capete. În acest exemplu ponderile sunt generate de proiecții aleatoare, nu învățate; prin urmare, nu atribui capetelor o funcție lingvistică fără dovezi suplimentare.

---
# 2. Tokenizarea textelor în română

Modelele Transformer prelucrează ID-uri de tokeni, nu cuvinte brute. Un tokenizer poate păstra un cuvânt întreg sau îl poate împărți în subtokeni. Rezultatul diferă între modele deoarece vocabularul și metoda de tokenizare au fost construite diferit.

## 2.1 Comparația tokenizatoarelor

Compar GPT-2, BERT multilingv și BERT românesc pe aceleași propoziții. Numărul de tokeni este specific textelor și tokenizatoarelor testate; nu este o măsură universală a calității modelului.

In [ ]:
tokenizer_names = {
    "GPT-2": "gpt2",
    "BERT multilingv": "bert-base-multilingual-cased",
    "BERT românesc": "dumitrescustefan/bert-base-romanian-cased-v1",
}
texts = [
    "Banca a aprobat creditul clientului.",
    "Împrumuturile restructurate au dobânzi variabile.",
    "Știința și tehnologia schimbă comunitățile românești.",
]

tokenizers = {
    label: AutoTokenizer.from_pretrained(model_id)
    for label, model_id in tokenizer_names.items()
}
tokenization_rows = []
for label, tokenizer in tokenizers.items():
    for text_number, text in enumerate(texts, start=1):
        pieces = tokenizer.tokenize(text)
        tokenization_rows.append({
            "Tokenizer": label,
            "Text": text_number,
            "Număr tokeni": len(pieces),
            "Subtokeni": " | ".join(pieces),
        })
        print(f"{label:16} | text {text_number} | {len(pieces):2} tokeni | {pieces}")

tokenization_table = pd.DataFrame(tokenization_rows)
display(tokenization_table.pivot(index="Tokenizer", columns="Text", values="Număr tokeni"))

**Analiză de completat:** identific un cuvânt românesc care a fost împărțit în mai mulți subtokeni. Compar rezultatul între tokenizatoare și discut ce arată acesta despre vocabularul fiecăruia. Observ și marcajele de spațiu/subcuvânt fără să le confund cu tokeni independenți.

## 2.2 Costul relativ al tokenizării

GPT-2 este antrenat în principal pe text în engleză. Compar aceeași idee exprimată în română și engleză cu același tokenizer GPT-2 și calculez raportul tokenilor. Acest experiment simplu ilustrează de ce numărul de tokeni poate afecta lungimea de context și costul unor servicii tarifate per token; nu estimează singur costul unui serviciu anume.

In [ ]:
romanian_text = "Modelul analizează cuvinte românești și generează propoziții coerente."
english_text = "The model analyzes Romanian words and generates coherent sentences."
gpt2_tokenizer = tokenizers["GPT-2"]

romanian_ids = gpt2_tokenizer(romanian_text, add_special_tokens=False).input_ids
english_ids = gpt2_tokenizer(english_text, add_special_tokens=False).input_ids
token_ratio = len(romanian_ids) / len(english_ids)
print(f"Română: {len(romanian_ids)} tokeni")
print(f"Engleză: {len(english_ids)} tokeni")
print(f"Raport RO/EN: {token_ratio:.2f}")
print("Subtokeni RO:", gpt2_tokenizer.convert_ids_to_tokens(romanian_ids))
print("Subtokeni EN:", gpt2_tokenizer.convert_ids_to_tokens(english_ids))

**Întrebare:** ce limite are această comparație? Pentru o concluzie mai solidă, aș testa mai multe propoziții aliniate ca sens și aș raporta media/variația raportului, nu doar un singur exemplu.

## 2.3 Diacritice românești și Unicode

Literele `ș` și `ş` arată asemănător, dar folosesc caractere Unicode diferite: prima are virgulă dedesubt (comma below), a doua sedilă. Normalizarea NFC compune formele canonice ale unui caracter, dar nu transformă automat sedila în virgulă. Inspectez codurile Unicode și tokenizarea celor două forme.

In [ ]:
forms = ["ș", "ş", "ț", "ţ"]
for char in forms:
    print(
        repr(char),
        "NFC =", repr(unicodedata.normalize("NFC", char)),
        "cod =", f"U+{ord(char):04X}",
        "nume =", unicodedata.name(char),
    )

diacritic_texts = ["școală și țară", "şcoală şi ţară"]
for text in diacritic_texts:
    print(repr(text), "->", gpt2_tokenizer.tokenize(text))

assert unicodedata.normalize("NFC", "ş") == "ş"
assert "ș" != "ş" and "ț" != "ţ"

**Concluzie de completat:** notez dacă variantele au fost tokenizate diferit și de ce este important să păstrez diacriticele corecte în date, căutare și evaluare.

---
# 3. Generarea autoregresivă

Un model causal prezice distribuția următorului token pe baza contextului disponibil. După selectarea tokenului, acesta este adăugat la context, iar modelul repetă predicția. Folosesc GPT-2 ca exemplu de model decoder-only; generările sunt demonstrative și pot conține erori, repetiții sau afirmații neverificate.

## 3.1 Topul probabilităților pentru următorul token

Modelul produce logits pentru vocabular. Aplic Softmax pentru a obține probabilități și inspectez cei mai probabili zece tokeni pentru un prompt. Tokenii afișați reprezintă fragmente, nu neapărat cuvinte complete.

In [ ]:
gpt2_name = "gpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(gpt2_name)
gpt2_model = AutoModelForCausalLM.from_pretrained(gpt2_name).to(device)
gpt2_model.eval()

prompt = "The bank approved the"
inputs = gpt2_tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
    next_logits = gpt2_model(**inputs).logits[0, -1]
next_probabilities = torch.softmax(next_logits, dim=-1)
top_probabilities, top_ids = torch.topk(next_probabilities, k=10)

top_rows = []
for token_id, probability in zip(top_ids.tolist(), top_probabilities.tolist()):
    top_rows.append({
        "ID": token_id,
        "Token decodat": gpt2_tokenizer.decode([token_id]),
        "Probabilitate": probability,
    })
display(pd.DataFrame(top_rows))
print("Suma probabilităților întregului vocabular:", float(next_probabilities.sum()))
assert torch.isclose(next_probabilities.sum(), torch.tensor(1.0, device=device), atol=1e-5)

**Interpretare de completat:** notez tokenul cu probabilitatea cea mai mare și observ dacă alternativele au sens în context. Probabilitatea modelului exprimă preferința sa condiționată de prompt, nu certitudinea că textul este adevărat.

## 3.2 Implementarea buclei de generare

Implementez decodarea greedy și sampling cu temperatură. Temperatura mică face distribuția mai concentrată; temperatura mare o aplatizează. Samplingul este aleatoriu, în timp ce greedy alege tokenul cu probabilitatea cea mai mare la fiecare pas. Fixez sămânța pentru reproductibilitate în aceeași configurație software.

In [ ]:
def generate_stepwise(model, tokenizer, text, max_new_tokens=40,
                      temperature=1.0, greedy=False):
    if max_new_tokens < 0:
        raise ValueError("max_new_tokens trebuie să fie nenegativ.")
    if not greedy and temperature <= 0:
        raise ValueError("Temperatura pentru sampling trebuie să fie pozitivă.")

    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
    for _ in range(max_new_tokens):
        with torch.inference_mode():
            logits = model(input_ids).logits[0, -1]
        if greedy:
            next_id = torch.argmax(logits).view(1, 1)
        else:
            probabilities = torch.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probabilities, num_samples=1).view(1, 1)
        input_ids = torch.cat((input_ids, next_id), dim=1)
        if tokenizer.eos_token_id is not None and next_id.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


generation_prompt = "The bank approved the"
greedy_text = generate_stepwise(
    gpt2_model, gpt2_tokenizer, generation_prompt, greedy=True
)
print("Greedy:\n", greedy_text)

for temperature in (0.3, 0.8, 1.3):
    torch.manual_seed(7)
    sampled_text = generate_stepwise(
        gpt2_model,
        gpt2_tokenizer,
        generation_prompt,
        temperature=temperature,
    )
    print(f"\nT = {temperature}:\n{sampled_text}")

**Tabel de observații — completez după ce citesc textele generate:**

| Metodă | Observații despre varietate, coerență și repetiții |
|---|---|
| Greedy | ... |
| T = 0.3 | ... |
| T = 0.8 | ... |
| T = 1.3 | ... |

Compar rezultatele observate, nu presupun că o anumită temperatură produce întotdeauna text mai bun. Generarea lingvistic fluentă nu garantează corectitudinea factuală.

## 3.3 Generarea textului în română (opțional)

GPT-2 de mai sus este un model general cu tokenizer asociat modelului său. Pentru a compara generarea într-un context românesc, încarc separat `readerbench/RoGPT2-base`. Această celulă este opțională deoarece descarcă un model suplimentar și consumă memorie.

In [ ]:
romanian_gpt_name = "readerbench/RoGPT2-base"
romanian_gpt_tokenizer = AutoTokenizer.from_pretrained(romanian_gpt_name)
romanian_gpt_model = AutoModelForCausalLM.from_pretrained(romanian_gpt_name).to(device)
romanian_gpt_model.eval()

romanian_prompt = "Banca a aprobat"
romanian_generation = generate_stepwise(
    romanian_gpt_model,
    romanian_gpt_tokenizer,
    romanian_prompt,
    max_new_tokens=40,
    temperature=0.8,
)
print(romanian_generation)

**Interpretare de completat:** evaluez gramatica, continuitatea și eventualele repetiții ale textului generat. Separ observațiile lingvistice de verificarea factuală; nu tratez modelul ca sursă de fapte.

---
# 4. Reprezentări contextuale și atenția BERT

BERT produce pentru fiecare token o reprezentare care depinde de contextul întregii propoziții. În această secțiune compar vectorul asociat cuvântului „bancă” în contexte financiare și în contexte despre un obiect de șezut. Compararea vectorilor este un indiciu numeric, nu o demonstrație completă a înțelegerii semantice.

## 4.1 Același cuvânt în contexte diferite

Folosesc tokenizatorul românesc fast pentru a obține offset-urile caracterelor și apoi fac media vectorilor subtokenilor care acoperă cuvântul țintă. Compar reprezentările embedding-ului de intrare (`hidden_states[0]`) cu cele ale ultimului strat.

In [ ]:
romanian_bert_name = "dumitrescustefan/bert-base-romanian-cased-v1"
bert_tokenizer = AutoTokenizer.from_pretrained(romanian_bert_name, use_fast=True)
bert_model = AutoModel.from_pretrained(
    romanian_bert_name,
    output_hidden_states=True,
).to(device)
bert_model.eval()

bank_sentences = [
    "Am deschis un cont la bancă.",
    "Banca a aprobat cererea pentru un credit.",
    "Ne-am așezat pe banca din parc.",
    "Banca de lemn era lângă fereastră.",
]
encoded = bert_tokenizer(
    bank_sentences,
    return_tensors="pt",
    padding=True,
    truncation=True,
    return_offsets_mapping=True,
).to(device)
offsets = encoded.pop("offset_mapping").cpu().numpy()
with torch.inference_mode():
    bert_outputs = bert_model(**encoded)

def target_token_vector(text, text_offsets, token_vectors, target="bancă"):
    start = text.casefold().find(target.casefold())
    if start < 0:
        raise ValueError(f"Cuvântul țintă {target!r} nu apare în text.")
    end = start + len(target)
    positions = [
        index
        for index, (left, right) in enumerate(text_offsets)
        if right > left and left < end and right > start
    ]
    if not positions:
        raise ValueError("Tokenizerul nu a asociat offset-uri cuvântului țintă.")
    return token_vectors[positions].mean(dim=0)


def cosine_similarity_matrix(vectors):
    vectors = torch.stack(vectors)
    vectors = torch.nn.functional.normalize(vectors, p=2, dim=-1)
    return (vectors @ vectors.T).cpu().numpy()


attention_mask = encoded["attention_mask"].bool()
layer_indices = {"Intrare (stratul 0)": 0, "Ultimul strat": len(bert_outputs.hidden_states) - 1}
similarities = {}
for label, layer_index in layer_indices.items():
    vectors = [
        target_token_vector(
            sentence,
            offsets[row, :int(attention_mask[row].sum().item())],
            bert_outputs.hidden_states[layer_index][row, :int(attention_mask[row].sum().item())],
        )
        for row, sentence in enumerate(bank_sentences)
    ]
    similarities[label] = cosine_similarity_matrix(vectors)
    print(f"{label}:\n", similarities[label])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, (label, matrix) in zip(axes, similarities.items()):
    image = axis.imshow(matrix, vmin=0, vmax=1, cmap="viridis")
    axis.set_title(label)
    axis.set_xticks(range(len(bank_sentences)), labels=["financiar 1", "financiar 2", "parc 1", "parc 2"], rotation=30)
    axis.set_yticks(range(len(bank_sentences)), labels=["financiar 1", "financiar 2", "parc 1", "parc 2"])
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            axis.text(col, row, f"{matrix[row, col]:.2f}", ha="center", va="center", color="white")
fig.colorbar(image, ax=axes, label="Similaritate cosinus")
fig.tight_layout()
plt.show()

**Analiză de completat:** compar similaritățile pentru perechile cu sens financiar (1–2) și pentru perechile dintre sensuri diferite (de exemplu 1–3). Verific dacă relația observată se schimbă între stratul de intrare și ultimul strat. Discut limitele: o valoare cosinus nu este o etichetă de sens și rezultatul depinde de propozițiile alese.

## 4.2 Vizualizarea atenției cu BertViz

BERT este bidirecțional, deci atenția sa nu folosește masca cauzală folosită de GPT-2. Vizualizarea de mai jos arată distribuții de atenție pentru o propoziție scurtă și permite explorarea straturilor și capetelor. Ponderea atenției descrie o parte din calculul modelului, dar nu trebuie interpretată automat ca explicație cauzală sau semnificație lingvistică.

In [ ]:
from bertviz import head_view

attention_model = AutoModel.from_pretrained(
    romanian_bert_name,
    output_attentions=True,
    attn_implementation="eager",
).to(device)
attention_model.eval()
attention_text = "Banca a aprobat creditul clientului."
attention_inputs = bert_tokenizer(attention_text, return_tensors="pt").to(device)
with torch.inference_mode():
    attention_outputs = attention_model(**attention_inputs)
tokens = bert_tokenizer.convert_ids_to_tokens(attention_inputs["input_ids"][0])
print("Tokeni:", tokens)
print("Număr de straturi:", len(attention_outputs.attentions))
print("Capete în primul strat:", attention_outputs.attentions[0].shape[1])
head_view(attention_outputs.attentions, tokens)

**Explorare:** selectez câteva capete și notez perechi de tokeni cu ponderi vizibile. Compar tiparele între capete, apoi formulez o interpretare prudentă. Un cap poate distribui atenția după tipare diferite, iar o singură hartă nu este suficientă pentru a explica predicția modelului.

---
# 5. Concluzii și răspunsuri

Răspund după executarea celulelor și folosesc valorile efectiv obținute.

1. **Atenție:** descriu pe scurt pașii de la scorurile $QK^T/\sqrt{d_k}$ la vectorul contextualizat și explic rolul lui V.
2. **Stabilitate numerică:** explic de ce scăderea maximului păstrează Softmax și previne overflow-ul.
3. **Mască cauzală:** descriu care poziții sunt excluse și legătura cu predicția autoregresivă.
4. **Multi-head:** explic de ce mai multe proiecții pot învăța tipare diferite și de ce exemplul cu ponderi aleatoare nu demonstrează funcții lingvistice.
5. **Tokenizare:** prezint cel puțin un rezultat măsurat pentru un text românesc și o limitare a comparației.
6. **Generare:** compar greedy și sampling la temperaturi diferite pe baza textelor produse; menționez de ce fluența nu implică adevăr factual.
7. **Reprezentări contextuale:** compar valorile cosinus pentru sensuri similare și diferite și notez o limitare a experimentului.
8. **Atenție BERT:** notez o observație din vizualizare și explic de ce atenția nu este, singură, o explicație completă.

### Reflecție personală

**Ce mi s-a părut cel mai important în laborator:** ...

**Ce rezultat m-a surprins și de ce:** ...

**O limitare a experimentelor și o îmbunătățire posibilă:** ...

### Verificare finală

- [ ] Toate celulele necesare au fost rulate în ordine.
- [ ] Interpretările descriu rezultatele obținute în această rulare.
- [ ] Am separat observațiile măsurate de ipoteze și am evitat să tratez textul generat ca fapt verificat.
- [ ] Am notat orice celulă opțională pe care am omis-o și motivul.